# core.dom.summary.object_utils

Utilities for table objects, recursive row groups and list items, and near-duplicate image filtering.

The DOM summary notebook supplies model responses and concurrency controls; this module does not import the DOM summarizer.

In [ ]:
# | default_exp core.dom.summary.object_utils

In [ ]:
# | export
import asyncio
from contextlib import nullcontext
from contextvars import ContextVar
from dataclasses import dataclass
from functools import lru_cache, partial, wraps
from hashlib import sha256
from html.parser import HTMLParser
from io import BytesIO
import json
import os
from pathlib import Path
import re
from threading import Lock
import warnings

from markdown_it import MarkdownIt
from PIL import Image, ImageOps


## Near-duplicate image filtering

Reference snapshots, cached image fingerprints, and skip metadata are shared throughout a run. Callers may supply an async semaphore and project root; model requests remain in the DOM summarizer.

In [ ]:
# | export
_IGNORED_IMAGE_ASPECT_TOLERANCE = 0.02
_IGNORED_IMAGE_HASH_DISTANCE = 4
_IGNORED_IMAGE_RGB_DIFFERENCE = 0.03
_IGNORED_IMAGE_SIZE = (64, 64)
_IMAGE_RASTER_SUFFIXES = frozenset({'.png', '.jpg', '.jpeg', '.gif', '.webp', '.bmp',
                                   '.tif', '.tiff', '.ico', '.ppm', '.pgm', '.pbm', '.avif'})
_ignored_images = ContextVar('ignored_images', default=None)
_ignored_image_warning_paths = set()
_ignored_image_warning_lock = Lock()


@dataclass(frozen=True)
class _IgnoredImageSnapshot:
    references: tuple
    signature: str


def _image_fingerprint(image):
    """Describe oriented static artwork on both backgrounds, including alpha."""
    if image.format not in {None, 'PNG', 'JPEG', 'GIF', 'WEBP', 'BMP', 'TIFF', 'ICO', 'PPM', 'DIB', 'AVIF'}:
        return None
    if getattr(image, 'is_animated', False):
        return None
    oriented = ImageOps.exif_transpose(image).convert('RGBA')
    backgrounds = []
    for color in ('black', 'white'):
        composite = Image.new('RGBA', oriented.size, color)
        composite.alpha_composite(oriented)
        rgb = composite.convert('RGB')
        gray = tuple(rgb.resize((9, 8), Image.Resampling.LANCZOS).convert('L').tobytes())
        difference_hash = sum(1 << (y * 8 + x) for y in range(8) for x in range(8)
                              if gray[y * 9 + x] > gray[y * 9 + x + 1])
        backgrounds.append((difference_hash,
                            rgb.resize(_IGNORED_IMAGE_SIZE, Image.Resampling.LANCZOS).tobytes()))
    return oriented.width / oriented.height, tuple(backgrounds)


@lru_cache(maxsize=512)
def _cached_image_file_info(path, size, mtime_ns, content_digest=None):
    """Cache immutable descriptors, content digests and decode failures by stat."""
    digest = None
    try:
        content = Path(path).read_bytes()
        digest = sha256(content).hexdigest()
        with Image.open(BytesIO(content)) as image:
            return digest, _image_fingerprint(image), None
    except (OSError, ValueError, SyntaxError, Image.DecompressionBombError) as exc:
        return digest, None, str(exc)


def _image_file_info(path, verify_content=False):
    try:
        info = path.stat()
        # References are rehashed per run so preserved timestamps cannot hide edits.
        digest = sha256(path.read_bytes()).hexdigest() if verify_content else None
        return _cached_image_file_info(str(path), info.st_size, info.st_mtime_ns, digest)
    except OSError as exc:
        return None, None, str(exc)


def _warn_ignored_image(path, error):
    with _ignored_image_warning_lock:
        if str(path) in _ignored_image_warning_paths:
            return
        _ignored_image_warning_paths.add(str(path))
    warnings.warn(f'Cannot read ignored-image reference {path}: {error}', RuntimeWarning, stacklevel=2)


def _load_ignored_image_snapshot(root=None):
    """Freeze the reference set and content-based checkpoint signature for a run."""
    configured = os.getenv('PROJ_ROOT')
    if configured:
        root = Path(configured).expanduser().resolve()
    elif root is None:
        root = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (path / 'pyproject.toml').is_file()), None)
    directory = root / 'assets' / 'images_ignored' if root is not None else None
    settings = {'matcher_version': 1, 'directory': str(directory),
                'aspect_tolerance': _IGNORED_IMAGE_ASPECT_TOLERANCE,
                'hash_distance': _IGNORED_IMAGE_HASH_DISTANCE,
                'rgb_difference': _IGNORED_IMAGE_RGB_DIFFERENCE,
                'size': _IGNORED_IMAGE_SIZE}
    references, files, manifest = [], [], []
    if directory is not None:
        try:
            files = sorted(p for p in directory.rglob('*')
                           if p.suffix.lower() in _IMAGE_RASTER_SUFFIXES and p.is_file())
        except OSError as exc:
            _warn_ignored_image(directory, exc)
    for path in files:
        digest, fingerprint, error = _image_file_info(path.resolve(), verify_content=True)
        manifest.append((str(path.relative_to(directory)), digest, error))
        if error is not None:
            _warn_ignored_image(path, error)
        if fingerprint is not None:
            references.append((str(path.resolve()), fingerprint))
    signature = sha256(json.dumps((settings, manifest), sort_keys=True).encode()).hexdigest()
    return _IgnoredImageSnapshot(tuple(references), signature)


async def _image_io(function, *args, semaphore=None):
    """Bound decoding threads and await their cleanup even on cancellation."""
    async with (semaphore if semaphore is not None else nullcontext()):
        work = asyncio.get_running_loop().run_in_executor(None, partial(function, *args))
        try:
            return await asyncio.shield(work)
        except asyncio.CancelledError:
            while not work.done():
                try:
                    await asyncio.shield(work)
                except asyncio.CancelledError:
                    continue
                except Exception:
                    break
            if work.done() and not work.cancelled():
                work.exception()
            raise


async def _prepare_ignored_images(*, semaphore=None, root=None):
    snapshot = _ignored_images.get()
    return snapshot if snapshot is not None else await _image_io(_load_ignored_image_snapshot, root, semaphore=semaphore)


def _with_ignored_images(function, *, get_semaphore=None, get_root=None):
    """Share one reference snapshot through recursive and concurrent calls."""
    @wraps(function)
    async def wrapped(*args, **kwargs):
        if _ignored_images.get() is not None:
            return await function(*args, **kwargs)
        snapshot = await _prepare_ignored_images(
            semaphore=get_semaphore() if get_semaphore is not None else None,
            root=get_root() if get_root is not None else None)
        token = _ignored_images.set(snapshot)
        try:
            return await function(*args, **kwargs)
        finally:
            _ignored_images.reset(token)
    return wrapped


def _image_fingerprints_match(left, right):
    left_aspect, left_backgrounds = left
    right_aspect, right_backgrounds = right
    if abs(left_aspect - right_aspect) > _IGNORED_IMAGE_ASPECT_TOLERANCE * max(left_aspect, right_aspect) + 1e-12:
        return False
    for (left_hash, left_rgb), (right_hash, right_rgb) in zip(left_backgrounds, right_backgrounds):
        if (left_hash ^ right_hash).bit_count() > _IGNORED_IMAGE_HASH_DISTANCE:
            return False
        difference = sum(abs(a - b) for a, b in zip(left_rgb, right_rgb)) / (255 * len(left_rgb))
        if difference > _IGNORED_IMAGE_RGB_DIFFERENCE + 1e-12:
            return False
    return True


def _match_ignored_image(image_link, snapshot):
    link = str(image_link)
    # URLs are left to the existing provider path; never fetch them for matching.
    if re.match(r'^[a-zA-Z][a-zA-Z0-9+.-]*://', link) or link.startswith('//'):
        return None
    path = Path(link).expanduser()
    if path.suffix.lower() not in _IMAGE_RASTER_SUFFIXES:
        return None
    _, fingerprint, _ = _image_file_info(path.resolve())
    if fingerprint is None:
        return None
    return next((reference for reference, candidate in snapshot.references
                 if _image_fingerprints_match(fingerprint, candidate)), None)


async def _skip_ignored_image(node, image_link, summary_key=None, *, semaphore=None, root=None):
    """Refresh skip metadata; source decode failures retain provider behavior."""
    node.pop('ignored_reference', None)
    node.pop('summary_ignored', None)
    snapshot = _ignored_images.get()
    if snapshot is None:
        snapshot = await _prepare_ignored_images(semaphore=semaphore, root=root)
    if not snapshot.references:
        return False
    reference = await _image_io(_match_ignored_image, image_link, snapshot, semaphore=semaphore)
    if reference is None:
        return False
    node['ignored_reference'] = reference
    if summary_key is not None:
        node['summary_ignored'] = True
        node[summary_key] = ''
    return True

## Tables and recursive row objects

Preserve original table source while extracting Markdown or HTML rows and nesting merged HTML ranges. Each physical row appears once, and semantic Div wrappers mirror the hierarchy.

In [ ]:
# | export
def _inline_text(tokens):
    return ''.join(
        _inline_text(t.children) if t.children else
        t.content if t.type in {'text', 'code_inline', 'image'} else
        ' ' if t.type in {'softbreak', 'hardbreak'} else ''
        for t in tokens
    ).strip()


class _TableSpans(HTMLParser):
    """Locate outer HTML tables without rewriting their source."""
    def __init__(self, source):
        super().__init__(convert_charrefs=False)
        self.source = source
        self.offsets = [0]
        for line in source.splitlines(keepends=True):
            self.offsets.append(self.offsets[-1] + len(line))
        self.depth = 0
        self.start = None
        self.spans = []
        self.feed(source)
        self.close()
        if self.depth:
            raise ValueError('Unclosed HTML table')

    def _offset(self):
        line, col = self.getpos()
        return self.offsets[line - 1] + col

    def handle_starttag(self, tag, attrs):
        if tag == 'table':
            if self.depth == 0:
                self.start = self._offset()
            self.depth += 1

    def handle_endtag(self, tag):
        if tag == 'table' and self.depth:
            self.depth -= 1
            if self.depth == 0:
                end = self.source.index('>', self._offset()) + 1
                self.spans.append((self.start, end))


class _HTMLTableRows(HTMLParser):
    """Read only outer-table rows, retaining source slices and cell spans."""
    def __init__(self, source):
        super().__init__(convert_charrefs=True)
        self.source, self.rows = source, []
        self.offsets = [0]
        for line in source.splitlines(keepends=True):
            self.offsets.append(self.offsets[-1] + len(line))
        self.depth = 0
        self.section, self.section_id = '', 0
        self.row = self.cell = None
        self.feed(source)
        self.close()
        self._finish_row(len(source))

    def _offset(self):
        line, column = self.getpos()
        return self.offsets[line - 1] + column

    def _finish_cell(self, end):
        if self.cell is not None:
            self.cell['content'] = self.source[self.cell.pop('_a'):end]
            self.cell['text'] = ' '.join(''.join(self.cell.pop('_text')).split())
            self.row['cells'].append(self.cell)
            self.cell = None

    def _finish_row(self, end):
        if self.row is not None:
            self._finish_cell(end)
            self.row['_b'] = end
            self.row['is_header'] = self.section == 'thead' or (
                bool(self.row['cells']) and all(c['is_header'] for c in self.row['cells']))
            self.rows.append(self.row)
            self.row = None

    def handle_starttag(self, tag, attrs):
        offset = self._offset()
        if tag == 'table':
            self.depth += 1
        if self.depth == 1:
            if tag in {'thead', 'tbody', 'tfoot'}:
                self._finish_row(offset)
                self.section, self.section_id = tag, self.section_id + 1
            elif tag == 'tr':
                self._finish_row(offset)
                self.row = {'_a': offset, 'cells': [], '_section': self.section_id}
            elif tag in {'td', 'th'} and self.row is not None:
                self._finish_cell(offset)
                attrs = dict(attrs)
                def span(name):
                    try:
                        value = int(attrs.get(name, '1'))
                        return value if value > 0 or (name == 'rowspan' and value == 0) else 1
                    except (ValueError, TypeError):
                        return 1
                self.cell = {'_a': offset, '_text': [], 'is_header': tag == 'th',
                             'rowspan': span('rowspan'), 'colspan': span('colspan')}
        if self.cell is not None and tag in {'br', 'p', 'div', 'li', 'table', 'tr', 'td', 'th'}:
            self.cell['_text'].append(' ')
        if self.cell is not None and tag == 'img':
            self.cell['_text'].append(dict(attrs).get('alt') or '')

    def handle_endtag(self, tag):
        offset = self._offset()
        end = self.source.index('>', offset) + 1
        if self.depth == 1:
            if tag in {'td', 'th'}:
                self._finish_cell(end)
            elif tag == 'tr':
                self._finish_row(end)
            elif tag in {'thead', 'tbody', 'tfoot', 'table'}:
                self._finish_row(offset)
                self.section, self.section_id = '', self.section_id + 1
        if self.cell is not None and tag in {'p', 'div', 'li', 'table', 'tr', 'td', 'th'}:
            self.cell['_text'].append(' ')
        if tag == 'table':
            self.depth -= 1

    def handle_data(self, data):
        if self.cell is not None:
            self.cell['_text'].append(data)


def _table_group_ranges(ranges):
    """Coalesce identical/crossing spans while retaining contained subgroups."""
    ranges = sorted(set(ranges))
    parents = list(range(len(ranges)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    # Crossing intervals cannot be siblings without duplicating physical rows.
    # Collapse their connected components; compatible contained spans survive.
    for i, (start, end) in enumerate(ranges):
        for j in range(i + 1, len(ranges)):
            next_start, next_end = ranges[j]
            if next_start >= end:
                break
            if start < next_start < end < next_end:
                parents[root(j)] = root(i)
    bounds = {}
    for i, (start, end) in enumerate(ranges):
        component = root(i)
        low, high = bounds.get(component, (start, end))
        bounds[component] = (min(low, start), max(high, end))
    return sorted(set(bounds.values()), key=lambda span: (span[0], -span[1]))


def _table_row_records(obj, content_line_start, env=None):
    """Build rows and nested merged groups without modifying table source."""
    source = obj['content']
    html = source.lstrip().lower().startswith('<table')
    rows, ranges = [], []
    if html:
        rows = _HTMLTableRows(source).rows
        # Vertical spans stop at row-group boundaries, including rowspan=0.
        ends = {}
        for i, row in enumerate(rows):
            ends[row['_section']] = i + 1
        active = []
        for i, row in enumerate(rows):
            active = [cell for cell in active if cell['source_row'] - 1 + cell['rowspan'] > i]
            occupied = {col for cell in active
                        for col in range(cell['column'], cell['column'] + cell['colspan'])}
            cells = [dict(cell, inherited=True) for cell in active]
            column = 1
            for cell in row['cells']:
                while any(col in occupied for col in range(column, column + cell['colspan'])):
                    column += 1
                remaining = ends[row['_section']] - i
                cell['rowspan'] = min(cell['rowspan'] or remaining, remaining)
                cell.update(column=column, source_row=i + 1, inherited=False)
                cells.append(cell)
                if cell['rowspan'] > 1:
                    active.append(cell)
                    ranges.append((i, i + cell['rowspan']))
                occupied.update(range(column, column + cell['colspan']))
                column += cell['colspan']
            row['cells'] = sorted(cells, key=lambda cell: cell['column'])
    else:
        lines = source.splitlines(keepends=True)
        offsets = [0]
        for line in lines:
            offsets.append(offsets[-1] + len(line))
        tokens = MarkdownIt('commonmark', {'html': True}).enable('table').parse(source, env)
        row, header = None, False
        for token in tokens:
            if token.type == 'thead_open':
                header = True
            elif token.type == 'thead_close':
                header = False
            elif token.type == 'tr_open':
                start, end = token.map
                row = {'_a': offsets[start], '_b': offsets[end],
                       'cells': [], 'is_header': header}
                rows.append(row)
            elif token.type == 'inline' and row is not None:
                row['cells'].append({'content': token.content, 'text': _inline_text(token.children or []),
                                     'column': len(row['cells']) + 1, 'source_row': len(rows),
                                     'rowspan': 1, 'colspan': 1, 'is_header': header, 'inherited': False})
            elif token.type == 'tr_close':
                row = None

    def record(kind, identity, a, b, **fields):
        return dict(id=f"{obj['id']}.{identity}", type=kind, table_id=obj['id'],
                    source_file=obj['source_file'],
                    line_start=content_line_start + source[:a].count('\n'),
                    line_end=content_line_start + source[:b].rstrip('\n').count('\n'),
                    content=source[a:b], caption=obj['caption'], images=[], summary='',
                    children=[], **fields)

    records = [record('table_row', f'row.{i + 1}', row['_a'], row['_b'],
                      row_index=i + 1, is_header=row['is_header'], cells=row['cells'])
               for i, row in enumerate(rows)]
    headers = [row['cells'] for row in records if row['is_header']]
    children = []
    # Each frame holds the range end, next unassigned row, and output children.
    stack = [[len(rows), 0, children]]
    for start, end in _table_group_ranges(ranges):
        while stack[-1][0] <= start:
            group_end, position, output = stack.pop()
            output.extend(records[position:group_end])
        parent = stack[-1]
        parent[2].extend(records[parent[1]:start])
        group = record('table_row_group', f'group.{start + 1}-{end}',
                       rows[start]['_a'], rows[end - 1]['_b'],
                       row_start=start + 1, row_end=end)
        parent[2].append(group)
        parent[1] = end
        stack.append([end, start, group['children']])
    while stack:
        group_end, position, output = stack.pop()
        output.extend(records[position:group_end])
    return children, headers


def _row_summary_nodes(obj):
    """Mirror semantic children as Divs, leaving the original Pandoc c intact."""
    return [{'t': 'Div', 'c': [['', [], []], []], 'object': child,
             's': child['summary'], 'subnodes': _row_summary_nodes(child)}
            for child in obj.get('children', [])]

## Row summaries

Summarize physical rows and groups bottom-up. The caller supplies `ordered_map`, `get_semaphore`, and `summarize_response`, preserving its provider selection, concurrency limits, and cancellation behavior.

In [ ]:
# | export
def _has_object_summary(record):
    """Accept existing text and intentionally empty summaries of ignored figures."""
    summary = record.get('summary')
    return (isinstance(summary, str) and bool(summary.strip())) or (
        record.get('type') == 'figure' and record.get('summary_ignored') is True
        and summary == '')


async def _summarize_table_child_async(
    row, table, client, model, lang, *, ordered_map, get_semaphore, summarize_response,
    reuse_existing=False,
):
    """Summarize rows bottom-up using the caller's provider and bounded runtime."""
    try:
        async def summarize_child(index, child):
            return await _summarize_table_child_async(
                child, table, client, model, lang, ordered_map=ordered_map,
                get_semaphore=get_semaphore, summarize_response=summarize_response,
                reuse_existing=reuse_existing)
        await ordered_map(summarize_child, row['children'])
        if reuse_existing and _has_object_summary(row):
            return row
        payload = {'caption': table['caption'], 'headers': table['headers'],
                   'content': row['content']}
        if row['type'] == 'table_row':
            instruction = 'Summarize this table row, preserving column relationships and inherited merged-cell values.'
            payload.update(row_index=row['row_index'], is_header=row['is_header'], cells=row['cells'])
        else:
            instruction = ('Summarize this merged table row group using every immediate child row '
                           'or subgroup summary and inherited merged-cell values.')
            first_row = row
            while first_row['type'] == 'table_row_group':
                first_row = first_row['children'][0]
            payload.update(row_start=row['row_start'], row_end=row['row_end'],
                           inherited_cells=[cell for cell in first_row['cells'] if cell['inherited']])
            payload['child_summaries'] = [child['summary'] for child in row['children']]
        prompt = instruction + '\n' + json.dumps(payload, ensure_ascii=False)
        async with get_semaphore():
            row['summary'] = await summarize_response(
                client=client, content=prompt, model=model, role='user', lang=lang)
        if not isinstance(row['summary'], str) or not row['summary'].strip():
            raise ValueError('Provider returned an empty row summary')
        return row
    except Exception as exc:
        raise ValueError(f"Error summarizing {row['type']} {row['id']} in table {table['id']} at "
                         f"{row['source_file']}:{row['line_start']}-{row['line_end']}: {exc}") from exc

## Recursive list items

Lists expose ordered `list_item` children, with nested items under their parent item. Each item retains its source span, content, caption and numbering. Summaries flow from nested items to parents and then to the complete list. Semantic Div subnodes mirror these children using the same wrapper helper as table rows.

In [ ]:
# | export
def _list_item_records(obj, content_line_start, env=None):
    """Retain every Markdown list item once, nesting children under their owning item.

    Source slices include continuation blocks and nested lists. Per-list indexes
    and ordered-list starts remain available even when an item owns several lists.
    """
    source = obj['content']
    lines = source.splitlines(keepends=True)
    tokens = MarkdownIt('commonmark', {'html': True}).enable('table').parse(source, env)
    children, lists, items = [], [], []
    for token in tokens:
        if token.type in {'bullet_list_open', 'ordered_list_open'}:
            list_start = token.attrGet('start')
            lists.append({'type': 'ordered' if token.type == 'ordered_list_open' else 'bullet',
                          'start': list_start if list_start is not None else 1, 'index': 0})
        elif token.type in {'bullet_list_close', 'ordered_list_close'}:
            lists.pop()
        elif token.type == 'list_item_open':
            context = lists[-1]
            context['index'] += 1
            parent = items[-1] if items else obj
            siblings = parent['children'] if items else children
            start, end = token.map
            while end > start and not lines[end - 1].strip():
                end -= 1
            item = dict(id=f"{parent['id']}.item.{len(siblings) + 1}", type='list_item',
                        list_id=obj['id'], parent_id=parent['id'],
                        item_index=context['index'], list_type=context['type'],
                        list_start=context['start'],
                        ordinal=(context['start'] + context['index'] - 1
                                 if context['type'] == 'ordered' else None),
                        source_file=obj['source_file'],
                        line_start=content_line_start + start,
                        line_end=content_line_start + end - 1,
                        content=''.join(lines[start:end]), caption=obj['caption'],
                        images=[], summary='', children=[])
            siblings.append(item)
            items.append(item)
        elif token.type == 'list_item_close':
            items.pop()
    return children


async def _summarize_list_item_async(
    item, obj, client, model, lang, *, ordered_map, get_semaphore, summarize_response,
    reuse_existing=False,
):
    """Summarize nested items before their parent using the caller's bounded runtime."""
    try:
        async def summarize_child(index, child):
            return await _summarize_list_item_async(
                child, obj, client, model, lang, ordered_map=ordered_map,
                get_semaphore=get_semaphore, summarize_response=summarize_response,
                reuse_existing=reuse_existing)
        await ordered_map(summarize_child, item['children'])
        if reuse_existing and _has_object_summary(item):
            return item
        payload = {key: item[key] for key in (
            'id', 'list_id', 'parent_id', 'item_index', 'list_type', 'list_start',
            'ordinal', 'caption', 'content')}
        payload['child_summaries'] = [child['summary'] for child in item['children']]
        prompt = ('Summarize this list item using every immediate child item summary, '
                  'preserving its meaning, order and nesting.\n'
                  + json.dumps(payload, ensure_ascii=False))
        async with get_semaphore():
            item['summary'] = await summarize_response(
                client=client, content=prompt, model=model, role='user', lang=lang)
        if not isinstance(item['summary'], str) or not item['summary'].strip():
            raise ValueError('Provider returned an empty list item summary')
        return item
    except Exception as exc:
        raise ValueError(f"Error summarizing list_item {item['id']} in list {obj['id']} at "
                         f"{item['source_file']}:{item['line_start']}-{item['line_end']}: {exc}") from exc
